In [4]:
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import oasis.functions
from tqdm import tqdm
import oasis
import pandas as pd

def normalize_crosscorr(C):
    """
    Normalize a 2D cross-correlation matrix to range [-1, 1].
    
    Parameters:
        C (numpy.ndarray): Input 2D correlation matrix.
    Returns:
        numpy.ndarray: Normalized correlation matrix.
    """
    C_min = np.min(C)
    C_max = np.max(C)
    
    # Avoid division by zero if the matrix is constant
    if C_max == C_min:
        return np.zeros_like(C)
    
    C_norm = ((C - C_min) / (C_max - C_min)) * 2 - 1
    return C_norm

def calculate_cross_correlation_multiple_iterations(data, n_iterations=10, subsample_ratio=0.8):
    """
    Calculate cross-correlation matrix across multiple iterations by subsampling the data.
    
    Parameters:
        data (numpy.ndarray): Neural activity data with shape (n_cells, n_frames)
        n_iterations (int): Number of iterations for correlation calculation
        subsample_ratio (float): Fraction of data to use in each iteration (0-1)
    
    Returns:
        numpy.ndarray: Mean cross-correlation matrix across iterations
        numpy.ndarray: Standard deviation of cross-correlations across iterations
    """
    n_cells, n_frames = data.shape
    all_correlation_matrices = np.zeros((n_iterations, n_cells, n_cells))
    
    for i in tqdm(range(n_iterations), desc="Calculating correlations"):
        # Randomly subsample frames
        n_subsample = int(n_frames * subsample_ratio)
        subsample_indices = np.random.choice(n_frames, n_subsample, replace=False)
        
        # Calculate correlation on subsampled data
        subsampled_data = data[:, subsample_indices]
        correlation_matrix = np.corrcoef(subsampled_data)
        all_correlation_matrices[i] = correlation_matrix
    
    # Calculate mean and std of correlation matrices
    mean_correlation = np.mean(all_correlation_matrices, axis=0)
    std_correlation = np.std(all_correlation_matrices, axis=0)
    
    return mean_correlation, std_correlation


In [5]:
# Find the number of subfolders in the folder
folder_path = r'F:\inyoung\250605'
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]
num_subfolders = len(subfolders)

# Set parameters for correlation calculation
n_iterations = 1000  # Number of iterations
subsample_ratio = 0.9  # Use 80% of frames in each iteration

# Loop through the subfolders
for subfolder in subfolders:
    try:
        # Get the base path and record name
        basepath = subfolder
        rec_name = os.path.basename(subfolder)
        print(f"Processing folder: {rec_name}")
        
        # Find all .mat files in the subfolder
        mat_files = [f for f in os.listdir(basepath) if f.endswith('.mat')]
        
        if not mat_files:
            print(f"No .mat files found in {rec_name}")
            continue
            
        for mat_file in mat_files:
            try:
                mat_file_path = os.path.join(basepath, mat_file)
                print(f"Loading .mat file: {mat_file}")
                
                # Load the .mat file
                mat_data = sio.loadmat(mat_file_path)
                
                # Extract the 'data' field
                if 'data' not in mat_data:
                    print("No 'data' field found in this .mat file")
                    continue
                    
                data_struct = mat_data['data']
                print(f"Data type: {data_struct.dtype}")
                
                # Extract each field from the structure
                data_fields = {}
                for field_name in data_struct.dtype.names:
                    data_fields[field_name] = data_struct[0, 0][field_name]
                
                # Extract fields of interest
                filename = data_fields['filename']
                numFrames = data_fields['numFrames'][0][0] if data_fields['numFrames'].size > 0 else 0
                frameRate = data_fields['frameRate'][0][0] if data_fields['frameRate'].size > 0 else 0
                cellMasks = data_fields['cellMasks']
                DFF_final = data_fields['DFF']
                
                # Print summary information
                print(f"  Filename: {filename[0] if filename.size > 0 else 'Empty'}")
                print(f"  Number of frames: {numFrames}")
                print(f"  Frame rate: {frameRate}")
                n_cells = cellMasks.shape[1] if hasattr(cellMasks, 'shape') else 0
                print(f"  Number of cells detected: {n_cells}")
                
                if n_cells == 0 or numFrames == 0:
                    print("No cells or frames detected, skipping this file")
                    continue
                
                # Initialize arrays for spike data
                spikes = np.zeros([n_cells, numFrames])
                norm_spikes = np.zeros([n_cells, numFrames])
                
                # Calculate spikes for each cell
                print("Calculating spikes...")
                for c in tqdm(range(n_cells)):
                    try:
                        # Deconvolved spiking activity
                        g = oasis.functions.estimate_time_constant(DFF_final[c,:].copy(), 1)
                        _, spikes[c,:] = oasis.oasisAR1(DFF_final[c,:].copy(), g)
                        
                        # Normalize spikes
                        spikes_min = np.min(spikes[c])
                        spikes_max = np.max(spikes[c])
                        if spikes_max > spikes_min:  # Avoid division by zero
                            norm_spikes[c] = (spikes[c] - spikes_min) / (spikes_max - spikes_min)
                    except Exception as e:
                        print(f"Error processing cell {c}: {e}")
                        
                # plt.plot(DFF_final, color = 'red')
                # plt.plot(spikes, color = 'blud')
                # plt.plot(norm_spikes, color = 'green')
                
                # Save the processed data
                twop_dict = {
                    'filename': filename,
                    'numFrames': numFrames,
                    'frameRate': frameRate,
                    'cellMasks': cellMasks,
                    'DFF_final': DFF_final,
                    'spikes': spikes,
                    'norm_spikes': norm_spikes
                }
                
                # Plot spikes
                print("Plotting spike data...")
                fig, axs = plt.subplots(1, 2, figsize=(20, 6))
                
                im0 = axs[0].imshow(spikes, aspect='auto', cmap='hot', interpolation='nearest')
                axs[0].set_title(f'Spikes for {rec_name}')
                axs[0].set_xlabel('Frames')
                axs[0].set_ylabel('Cells')
                fig.colorbar(im0, ax=axs[0], label='Spike amplitude')
                
                im1 = axs[1].imshow(norm_spikes, aspect='auto', cmap='hot', interpolation='nearest')
                axs[1].set_title(f'Normalized spikes for {rec_name}')
                axs[1].set_xlabel('Frames')
                axs[1].set_ylabel('Cells')
                fig.colorbar(im1, ax=axs[1], label='Normalized spike amplitude')
                
                plt.tight_layout()
                spike_fig_path = os.path.join(basepath, f"{rec_name}_spikes.jpg")
                plt.savefig(spike_fig_path, dpi=300)
                plt.close()
                print(f"Saved spike plot to {spike_fig_path}")
                
                # Calculate cross-correlation with multiple iterations
                print(f"Calculating cross-correlations across {n_iterations} iterations...")
                mean_corr_dff, std_corr_dff = calculate_cross_correlation_multiple_iterations(
                    DFF_final, n_iterations=n_iterations, subsample_ratio=subsample_ratio
                )
                
                mean_corr_spikes, std_corr_spikes = calculate_cross_correlation_multiple_iterations(
                    spikes, n_iterations=n_iterations, subsample_ratio=subsample_ratio
                )
                
                # Normalize the mean correlation matrices
                norm_corr_dff = normalize_crosscorr(mean_corr_dff)
                norm_corr_spikes = normalize_crosscorr(mean_corr_spikes)
                
                # Plot the cross-correlation matrices
                plt.figure(figsize=(20, 6))
                
                plt.subplot(1, 3, 1)
                im1 = plt.imshow(mean_corr_dff, aspect='auto', cmap='coolwarm', interpolation='nearest', vmin=0, vmax=1)
                plt.colorbar(im1, label='Cross-correlation (DFF)')
                plt.title(f'Mean Cross-correlation (DFF)\n{n_iterations} iterations')
                plt.xlabel('Cells')
                plt.ylabel('Cells')
                
                plt.subplot(1, 3, 2)
                im2 = plt.imshow(mean_corr_spikes, aspect='auto', cmap='coolwarm', interpolation='nearest', vmin=0, vmax=1)
                plt.colorbar(im2, label='Cross-correlation (Spikes)')
                plt.title(f'Mean Cross-correlation (Spikes)\n{n_iterations} iterations')
                plt.xlabel('Cells')
                plt.ylabel('Cells')
                
                plt.subplot(1, 3, 3)
                im3 = plt.imshow(std_corr_dff, aspect='auto', cmap='viridis', interpolation='nearest')
                plt.colorbar(im3, label='Std Dev of Correlations')
                plt.title(f'Std Dev of Cross-correlations\n{n_iterations} iterations')
                plt.xlabel('Cells')
                plt.ylabel('Cells')
                
                plt.tight_layout()
                corr_fig_path = os.path.join(basepath, f"{rec_name}_cross_correlation_{n_iterations}iter.jpg")
                plt.savefig(corr_fig_path, dpi=300)
                plt.close()
                print(f"Saved correlation plot to {corr_fig_path}")
                
                # Save the correlation matrices
                corr_dict = {
                    'mean_corr_dff': mean_corr_dff,
                    'std_corr_dff': std_corr_dff,
                    'mean_corr_spikes': mean_corr_spikes,
                    'std_corr_spikes': std_corr_spikes,
                    'n_iterations': n_iterations,
                    'subsample_ratio': subsample_ratio
                }
                
                npy_filename = os.path.splitext(mat_file)[0] + f'_correlations_{n_iterations}iter.npy'
                npy_file_path = os.path.join(basepath, npy_filename)
                np.save(npy_file_path, corr_dict)
                print(f"Saved correlation data to {npy_filename}")
                
                csv_filename = os.path.splitext(mat_file)[0] + f'_correlations_{n_iterations}iter.csv'
                csv_file_path = os.path.join(basepath, csv_filename)
                pd.DataFrame(mean_corr_dff).to_csv(csv_file_path, index=False)
                print(f"Saved correlation data to {csv_filename}")
                
            except Exception as e:
                print(f"Error processing file {mat_file}: {e}")
    
    except Exception as e:
        print(f"Error processing folder {subfolder}: {e}")

print("Processing complete!")

Processing folder: B1_WS259.2_D90_org3-002
Loading .mat file: B1_data.mat
Data type: [('filename', 'O'), ('numFrames', 'O'), ('xPixels', 'O'), ('yPixels', 'O'), ('map_type', 'O'), ('frameRate', 'O'), ('avg_projection', 'O'), ('frame_F', 'O'), ('activity_map', 'O'), ('cellMasks', 'O'), ('raw_F', 'O'), ('normtype', 'O'), ('neuropil_F', 'O'), ('r_neuropil', 'O'), ('DFF_raw', 'O'), ('DFF', 'O')]
  Filename: B1.tif
  Number of frames: 4507
  Frame rate: 15.023206948701938
  Number of cells detected: 511
Calculating spikes...


  0%|          | 0/511 [00:00<?, ?it/s]c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\oasis\functions.py:817: FutureWarning: Beginning in SciPy 1.17, multidimensional input will be treated as a batch, not `ravel`ed. To preserve the existing behavior and silence this warning, `ravel` arguments before passing them to `toeplitz`.
  A = scipy.linalg.toeplitz(xc[np.arange(lags)],
C:\Users\jasmineyeo\AppData\Local\Temp\ipykernel_47336\1093152617.py:74: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  _, spikes[c,:] = oasis.oasisAR1(DFF_final[c,:].copy(), g)
100%|██████████| 511/511 [00:00<00:00, 2013.02it/s]


Plotting spike data...
Saved spike plot to F:\inyoung\250605\B1_WS259.2_D90_org3-002\B1_WS259.2_D90_org3-002_spikes.jpg
Calculating cross-correlations across 1000 iterations...


Calculating correlations: 100%|██████████| 1000/1000 [00:20<00:00, 49.28it/s]


Saved correlation plot to F:\inyoung\250605\B1_WS259.2_D90_org3-002\B1_WS259.2_D90_org3-002_cross_correlation_1000iter.jpg
Saved correlation data to B1_data_correlations_1000iter.npy
Saved correlation data to B1_data_correlations_1000iter.csv
Processing folder: batch2_d90_dorsal_WS185.1_org2-002
Loading .mat file: batch2_data.mat
Data type: [('filename', 'O'), ('numFrames', 'O'), ('xPixels', 'O'), ('yPixels', 'O'), ('map_type', 'O'), ('frameRate', 'O'), ('avg_projection', 'O'), ('frame_F', 'O'), ('activity_map', 'O'), ('cellMasks', 'O'), ('raw_F', 'O'), ('normtype', 'O'), ('neuropil_F', 'O'), ('r_neuropil', 'O'), ('DFF_raw', 'O'), ('DFF', 'O')]
  Filename: batch2.tif
  Number of frames: 4507
  Frame rate: 15.023206948701938
  Number of cells detected: 101
Calculating spikes...


  0%|          | 0/101 [00:00<?, ?it/s]c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\oasis\functions.py:817: FutureWarning: Beginning in SciPy 1.17, multidimensional input will be treated as a batch, not `ravel`ed. To preserve the existing behavior and silence this warning, `ravel` arguments before passing them to `toeplitz`.
  A = scipy.linalg.toeplitz(xc[np.arange(lags)],
C:\Users\jasmineyeo\AppData\Local\Temp\ipykernel_47336\1093152617.py:74: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  _, spikes[c,:] = oasis.oasisAR1(DFF_final[c,:].copy(), g)
100%|██████████| 101/101 [00:00<00:00, 1310.51it/s]

Plotting spike data...


Saved spike plot to F:\inyoung\250605\batch2_d90_dorsal_WS185.1_org2-002\batch2_d90_dorsal_WS185.1_org2-002_spikes.jpg
Calculating cross-correlations across 1000 iterations...


Calculating correlations:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Calculating correlations: 100%|██████████| 1000/1000 [00:03<00:00, 311.99it/s]


Saved correlation plot to F:\inyoung\250605\batch2_d90_dorsal_WS185.1_org2-002\batch2_d90_dorsal_WS185.1_org2-002_cross_correlation_1000iter.jpg
Saved correlation data to batch2_data_correlations_1000iter.npy
Saved correlation data to batch2_data_correlations_1000iter.csv
Processing complete!
